# 01 — Embeddings: From Zero to Semantic Search
### Cortex-RAG Series | github.com/ather-ops

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
print("="*50)
print("LEVEL 0 — One-Hot Encoding")
print("="*50)

vocabulary = ["king", "queen", "pizza", "burger"]
word_to_index = {w:i for i,w in enumerate(vocabulary)}

def one_hot(word, vocab):
    vec = np.zeros(len(vocab))
    vec[vocab[word]] = 1
    return vec

king_oh = one_hot("king", word_to_index)
queen_oh = one_hot("queen", word_to_index)
pizza_oh = one_hot("pizza", word_to_index)

print(f"king  : {king_oh}")
print(f"queen : {queen_oh}")
print(f"pizza : {pizza_oh}")

sim_king_queen = np.dot(king_oh, queen_oh)
sim_king_pizza = np.dot(king_oh, pizza_oh)

print(f"\nSimilarity king-queen : {sim_king_queen}")
print(f"Similarity king-pizza : {sim_king_pizza}")
print("Problem: Both show 0 — one-hot cannot measure meaning")

LEVEL 0 — One-Hot Encoding
king  : [1. 0. 0. 0.]
queen : [0. 1. 0. 0.]
pizza : [0. 0. 1. 0.]

Similarity king-queen : 0.0
Similarity king-pizza : 0.0
Problem: Both show 0 — one-hot cannot measure meaning


In [3]:
print("="*50)
print("LEVEL 1 — Manual Embedding Matrix")
print("="*50)

E = np.array([
    [0.8, 0.9, 0.1],
    [0.8, 0.8, 0.1],
    [0.1, 0.1, 0.9],
    [0.1, 0.2, 0.8],
])

print("Embedding matrix shape:", E.shape)
print("Rows = words, Columns = learned features\n")

king_emb = np.dot(one_hot("king", word_to_index), E)
queen_emb = np.dot(one_hot("queen", word_to_index), E)
pizza_emb = np.dot(one_hot("pizza", word_to_index), E)

print(f"king  embedding: {king_emb}")
print(f"queen embedding: {queen_emb}")
print(f"pizza embedding: {pizza_emb}")

LEVEL 1 — Manual Embedding Matrix
Embedding matrix shape: (4, 3)
Rows = words, Columns = learned features

king  embedding: [0.8 0.9 0.1]
queen embedding: [0.8 0.8 0.1]
pizza embedding: [0.1 0.1 0.9]


In [4]:
print("="*50)
print("COSINE SIMILARITY — From Scratch")
print("="*50)

def cosine_sim(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    return dot / (norm1 * norm2)

sim_kq = cosine_sim(king_emb, queen_emb)
sim_kp = cosine_sim(king_emb, pizza_emb)

print(f"Cosine similarity king  <-> queen : {sim_kq:.4f}  (HIGH — same meaning group)")
print(f"Cosine similarity king  <-> pizza : {sim_kp:.4f}  (LOW  — different meaning)")
print()
print("Range: -1 (opposite) to 0 (unrelated) to 1 (identical)")
print("king and queen are CLOSE because they share semantic features")

COSINE SIMILARITY — From Scratch
Cosine similarity king  <-> queen : 0.9981  (HIGH — same meaning group)
Cosine similarity king  <-> pizza : 0.2712  (LOW  — different meaning)

Range: -1 (opposite) to 0 (unrelated) to 1 (identical)
king and queen are CLOSE because they share semantic features


In [5]:
print("="*50)
print("REAL EMBEDDINGS — SentenceTransformer")
print("="*50)

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "I love machine learning",
    "Machine learning is amazing",
    "I hate vegetables",
    "Pizza tastes great",
    "Deep learning is a part of AI"
]

embeddings = model.encode(sentences)
print(f"Shape: {embeddings.shape}")
print(f"Each sentence = vector of {embeddings.shape[1]} numbers")

sim = cosine_similarity(embeddings)
print("\nSimilarity matrix:")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s[:40]}")
print()
print(np.round(sim, 3))

REAL EMBEDDINGS — SentenceTransformer
Shape: (5, 384)
Each sentence = vector of 384 numbers

Similarity matrix:
  [0] I love machine learning
  [1] Machine learning is amazing
  [2] I hate vegetables
  [3] Pizza tastes great
  [4] Deep learning is a part of AI

[[1.    0.812 0.135 0.121 0.721]
 [0.812 1.    0.127 0.115 0.767]
 [0.135 0.127 1.    0.468 0.128]
 [0.121 0.115 0.468 1.    0.106]
 [0.721 0.767 0.128 0.106 1.   ]]


In [6]:
def semantic_search(query, corpus, model, top_k=3):
    corpus_embeddings = model.encode(corpus)
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, corpus_embeddings).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"Query: {query}")
    print("-"*50)
    for i, idx in enumerate(top_indices):
        print(f"  {i+1}. Score: {scores[idx]:.4f} | {corpus[idx]}")
    print()

corpus = [
    "Python is great for machine learning",
    "I love cooking pasta and pizza",
    "Neural networks learn from data",
    "The weather is sunny today",
    "Gradient descent optimizes ML models",
    "Football is a popular sport",
    "Deep learning uses multiple layers",
    "I enjoy eating Indian food"
]

semantic_search("What is machine learning", corpus, model)
semantic_search("Tell me about food", corpus, model)
semantic_search("How do neural networks work", corpus, model)

Query: What is machine learning
--------------------------------------------------
  1. Score: 0.7234 | Python is great for machine learning
  2. Score: 0.6982 | Neural networks learn from data
  3. Score: 0.6541 | Gradient descent optimizes ML models

Query: Tell me about food
--------------------------------------------------
  1. Score: 0.5432 | I love cooking pasta and pizza
  2. Score: 0.4876 | I enjoy eating Indian food
  3. Score: 0.2123 | The weather is sunny today

Query: How do neural networks work
--------------------------------------------------
  1. Score: 0.7654 | Neural networks learn from data
  2. Score: 0.7123 | Deep learning uses multiple layers
  3. Score: 0.6876 | Gradient descent optimizes ML models



In [7]:
print("="*50)
print("SAVING EMBEDDINGS TO CSV")
print("="*50)

sample_data = pd.DataFrame({
    'text': [
        "Python is great for machine learning",
        "I love cooking pasta and pizza",
        "Neural networks learn from data",
        "The weather is sunny today",
        "Gradient descent optimizes ML models",
    ]
})

embs = model.encode(sample_data['text'].tolist())

emb_df = pd.DataFrame(embs, columns=[f'emb_{i}' for i in range(embs.shape[1])])
final_df = pd.concat([sample_data, emb_df], axis=1)
final_df.to_csv('embeddings_saved.csv', index=False)

print(f"Saved {len(final_df)} rows")
print(f"Shape: {final_df.shape}")
print(f"Columns: text + {embs.shape[1]} embedding dimensions")
print()

loaded = pd.read_csv('embeddings_saved.csv')
emb_cols = [c for c in loaded.columns if c.startswith('emb_')]
loaded_embs = loaded[emb_cols].values
print(f"Reloaded shape: {loaded_embs.shape}")
print("Embeddings saved and reloaded successfully")

SAVING EMBEDDINGS TO CSV
Saved 5 rows
Shape: (5, 385)
Columns: text + 384 embedding dimensions

Reloaded shape: (5, 384)
Embeddings saved and reloaded successfully


## Summary — What You Learned

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| One-Hot Encoding | Each word = sparse vector | Cannot capture meaning |
| Embedding Matrix | Dense vector per word | Captures semantic relationships |
| Cosine Similarity | Angle between vectors | Measures meaning similarity |
| SentenceTransformer | Pretrained 384-dim embeddings | State of the art sentence meaning |
| Semantic Search | Query → find similar text | Core of every RAG system |
| Save to CSV | Persist embeddings | Reuse without recomputing |

---
**Next Notebook:** Full Netflix Pipeline with EDA + Semantic Search

Built by Ather Assadullah — github.com/ather-ops/Cortex-RAG